# pandas: таблица объявлений краткосрочной аренды

Отдел аналитики «StayLocal» хранит объявления в CSV. В модуле 1 `predict` работал на парах чисел; здесь **сотни строк** и **несколько полей** на объявление — нужен `DataFrame`.

In [ ]:
from pathlib import Path
import pandas as pd


def find_listings_csv() -> Path:
    for p in (Path('listings.csv'), Path('../../data/listings.csv'), Path('../data/listings.csv')):
        if p.exists():
            return p.resolve()
    raise FileNotFoundError('listings.csv не найден')


LISTINGS_PATH = find_listings_csv()
df = pd.read_csv(LISTINGS_PATH)


## 1. Зачем таблица

Одна **строка** = одно объявление; один **столбец** = одно поле (вместимость, отзывы, район…). Выведите число строк и столбцов, первые 5 строк и список имён столбцов.

**Вопрос:** сколько объявлений в файле и сколько полей у каждой?

**Pitfall:** `print(df)` на больших данных заливает экран — для осмотра хватает `shape`, `head`, `columns`.

In [ ]:
n_rows, n_cols = None, None  # df.shape
# print(df.head())
# print(list(df.columns))
assert n_rows is not None and n_cols is not None
assert isinstance(n_rows, (int, float)) and n_rows > 50
assert n_cols >= 5
print(n_rows, n_cols)

## 2. dtypes: число vs текст

Вызовите `df.dtypes` (или `df.info()`). Числовые столбцы нужны для `fit`; `object` — строки/категории, их нельзя сразу подать в `LinearRegression`.

**How:** `df['accommodates'].dtype` vs `df['neighbourhood'].dtype`.

**Checkpoint:** почему `accommodates` не `object`?

In [ ]:
# print(df.dtypes)
accommodates_dtype = None  # str(df['accommodates'].dtype)
neighbourhood_dtype = None  # str(df['neighbourhood'].dtype)
assert accommodates_dtype is not None and neighbourhood_dtype is not None
assert 'object' not in str(accommodates_dtype).lower() or 'float' in str(accommodates_dtype).lower() or 'int' in str(accommodates_dtype).lower()
assert 'object' in str(neighbourhood_dtype).lower() or str(neighbourhood_dtype) == 'object'
print('accommodates:', accommodates_dtype, '| neighbourhood:', neighbourhood_dtype)

## 3. Одна ячейка через iloc

Вместимость первого объявления: `df.iloc[0]['accommodates']` (или `df.iloc[0, …]`). Запишите в `first_accommodates`.

**Зачем:** модель потом читает признаки **построчно**.

**Pitfall:** `iloc` — по **позиции** (0, 1, 2…), не по `listing_id`.

In [ ]:
first_accommodates = None
assert first_accommodates is not None
assert isinstance(first_accommodates, (int, float)) and first_accommodates > 0
print('первый listing, accommodates:', first_accommodates)

## 4. loc vs iloc — нюанс

Возьмите первые 3 строки столбцов `accommodates` и `price` двумя способами: `.loc` (имена) и `.iloc` (позиции). Значения должны совпасть.

**Why:** после фильтра индекс строк может быть не 0…n; `iloc[0]` — «первая видимая», `loc[0]` — «строка с меткой 0» (может отсутствовать).

In [ ]:
by_name = None  # .loc
by_pos = None  # .iloc
assert by_name is not None and by_pos is not None
assert by_name.shape == (3, 2) and by_pos.shape == (3, 2)
assert list(by_name.columns) == ['accommodates', 'price']
assert (by_name.values == by_pos.values).all()
print(by_name)
print(by_pos)

## 5. Признак, цель, служебный столбец

Для ML явно назовите роли:
- **цель** — что предсказываем (`price`);
- **признаки** — числа/категории для модели;
- **id** — служебный ключ, **не** признак.

Заполните переменные ниже.

In [ ]:
TARGET = ''
FEATURES = []
ID_COL = ''

assert TARGET in df.columns
assert TARGET not in FEATURES
assert ID_COL in df.columns
assert ID_COL not in FEATURES
assert 'accommodates' in FEATURES
assert set(FEATURES).issubset(df.columns)
assert df[TARGET].dtype.kind in 'iuf'

## 6. value_counts: районы

Сколько объявлений в каждом `neighbourhood`? Используйте `value_counts()` (частоты категорий — база для EDA и фильтров).

Запишите число **уникальных** зон в `n_neighbourhoods`.

In [ ]:
neighbourhood_counts = None  # df['neighbourhood'].value_counts()
# print(neighbourhood_counts)
n_neighbourhoods = None
assert neighbourhood_counts is not None
assert n_neighbourhoods is not None
assert isinstance(n_neighbourhoods, (int, float)) and 2 <= n_neighbourhoods <= 20
print('уникальных neighbourhood:', n_neighbourhoods)
print(neighbourhood_counts)

## 7. Checkpoint: почему не list of dicts

В модуле 1 данные жили в списках. Здесь — таблица.

Напишите 2–3 предложения в `WHY_DATAFRAME`: зачем `DataFrame` вместо `list[dict]` для сотен объявлений (фильтр, столбец целиком, dtypes, готовность к `sklearn`).

**How:** подумайте про `df['accommodates']` vs цикл по словарям.

In [ ]:
WHY_DATAFRAME = ''
assert len(WHY_DATAFRAME) > 40
assert 'list' in WHY_DATAFRAME.lower() or 'слов' in WHY_DATAFRAME.lower() or 'dataframe' in WHY_DATAFRAME.lower() or 'таблиц' in WHY_DATAFRAME.lower()
print(WHY_DATAFRAME)

## 8. Расширение: объявление по id

Найдите цену объявления `L0001` через `.loc` и маску по `listing_id`.

**Зачем:** id нужен для поиска строки, но в `X` для `fit` его не кладут.

In [ ]:
price_l0001 = None
assert price_l0001 is not None
assert isinstance(price_l0001, (int, float)) and price_l0001 > 0
print('L0001 price:', price_l0001)